In [1]:
%%capture
%pip install -qU langchain
%pip install -qU langchain-openai
%pip install -qU langchain-community
%pip install python-dotenv

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.chains import (
    LLMChain,
    SimpleSequentialChain,
    SequentialChain,
    TransformChain,
    ConversationChain,
    LLMRouterChain,
    MultiPromptChain
)
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain.callbacks import StdOutCallbackHandler

In [4]:
load_dotenv()

True

### Simple LLM chain

In [ ]:
llm = ChatOpenAI(temperature=0.7)

prompt = PromptTemplate(
    input_variables=["product"],
    template="Write a creative product description for {product}. Make it appealing and under 100 words."
)

# Run the chain
chain = prompt | llm
result = chain.invoke("wireless noise-canceling headphones")

print(f"Result: {result}\n")

Result: content="Experience a new level of immersive sound with our wireless noise-canceling headphones. Say goodbye to distractions and hello to crystal-clear audio quality. With up to 30 hours of battery life, you can enjoy your favorite music, podcasts, and calls all day long. The sleek and comfortable design ensures a perfect fit for any lifestyle. Whether you're commuting, working out, or just relaxing at home, these headphones will elevate your listening experience to new heights. Upgrade to the ultimate in wireless audio technology today." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 28, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CN6FoIuoqeKc0G8QE19YIx

### Sequencial Chain

In [10]:
outline_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Create a brief outline for a short story about {topic}. Include 3 main points."
)
outline_chain = LLMChain(llm=llm, prompt=outline_prompt, output_key="outline")

# Second chain: Write the story based on outline
story_prompt = PromptTemplate(
    input_variables=["outline"],
    template="Based on this outline: {outline}\n\nWrite a short story (200 words)."
)
story_chain = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

# Third chain: Create a title
title_prompt = PromptTemplate(
    input_variables=["story"],
    template="Based on this story: {story}\n\nCreate a catchy title for it."
)
title_chain = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

# Combine chains
overall_chain = SequentialChain(
    chains=[outline_chain, story_chain, title_chain],
    input_variables=["topic"],
    output_variables=["outline", "story", "title"],
    verbose=True
)

# Run the chain
result = overall_chain.invoke({"topic": "a robot learning to paint"})

print(f"\nOutline: {result['outline']}\n")
print(f"Story: {result['story']}\n")
print(f"Title: {result['title']}\n")
   



> Entering new SequentialChain chain...

> Finished chain.

Outline: I. Introduction
- Introduce the main character, a robot named Arti, who is programmed to learn and perform various tasks.
- Arti's creator, a renowned artist, decides to teach Arti how to paint in order to explore the intersection of technology and art.

II. Learning to paint
- Arti initially struggles to understand the concept of creativity and self-expression.
- Through trial and error, Arti begins to experiment with different colors, strokes, and techniques.
- With the guidance of its creator, Arti starts to develop its own unique style and vision.

III. The art show
- Arti's creator decides to showcase Arti's paintings in a gallery alongside their own work.
- Attendees are skeptical at first, but are amazed by the depth and emotion conveyed in Arti's paintings.
- The success of Arti's debut as an artist highlights the potential of artificial intelligence in the world of art.

Story: Arti was a robot unlike any o

### Transform chain

In [11]:
def transform_func(inputs: dict) -> dict:
    """Custom transformation: clean and format text"""
    text = inputs["text"]
    # Remove extra whitespace
    clean_text = " ".join(text.split())
    # Convert to uppercase for emphasis
    emphasized = clean_text.upper()
    # Add word count
    word_count = len(clean_text.split())
    
    return {
        "original": text,
        "cleaned": clean_text,
        "emphasized": emphasized,
        "word_count": word_count
    }

# Create transform chain
transform_chain = TransformChain(
    input_variables=["text"],
    output_variables=["original", "cleaned", "emphasized", "word_count"],
    transform=transform_func
)

# Use with LLM chain
llm = ChatOpenAI(temperature=0.5)

analysis_prompt = PromptTemplate(
    input_variables=["cleaned", "word_count"],
    template="Analyze this {word_count}-word text: '{cleaned}'. Provide sentiment and key themes."
)

analysis_chain = LLMChain(llm=llm, prompt=analysis_prompt)

# Combine chains
overall_chain = SequentialChain(
    chains=[transform_chain, analysis_chain],
    input_variables=["text"],
    verbose=True
)

# Run the chain
result = overall_chain({
    "text": "  LangChain   is   amazing   for   building   LLM   applications!  "
})

print(f"Result: {result}\n")

ValidationError: 1 validation error for SequentialChain
  Value error, Chain returned keys that already exist: {'text'} [type=value_error, input_value={'chains': [TransformChai...text'], 'verbose': True}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error